# Process CSDA Evaluation Sites AOIs to a `GeoJSON` polygon

Paul Montesano, PhD  
April 2026

In [1]:
import pandas as pd
import geopandas as gpd
from urllib.parse import quote
import numpy as np

import sys

sys.path.append('/projects/code/csda_summaries/lib')
sys.path.append('/home/pmontesa/code/csda_summaries/lib')

import siteslib

from datetime import datetime

## Evaluation Sites

#### Set default site AOI extent size (either a radius, or a box side - depending on `aoi_type` field)

In [2]:
BUF_KM = 3

In [3]:
# Define your spreadsheet ID and sheet name
SPREADSHEET_ID = '13MrpqFtAOqQY9WdW9lHNsqjCbG-e3VQkEDbHOGIKa6k'

#### Read in sites database

In [4]:
SHEET_NAME = 'Evaluation Sites' # The name of the specific tab

# 2. Encode the sheet name for safe use in a URL
ENCODED_SHEET_NAME = quote(SHEET_NAME)

# 3. Construct the full URL using the gviz/tq endpoint
url = f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/gviz/tq?tqx=out:csv&sheet={ENCODED_SHEET_NAME}"

# 4. Use pandas to read the CSV data directly from the URL
try:
    sites = pd.read_csv(url)
    sites['Site Name'] = sites['Site Name abbrev'].str.rstrip()
    #sites = sites[sites['Site Name'] != 'Sicily'] # Sicily site is same as Catania
    
except Exception as e:
    print(f"An error occurred: {e}")

# Get only Priority Sites - used for defining CSDA sites for now
sites = sites[sites['Priority Level'] != 'unknown']

# # Get only Priority Sites = high
# sites = sites[sites['Priority Level'] == 'high']

# Get the list of columns to drop
cols_to_drop = sites.columns[sites.columns.str.contains('Unnamed')]
sites = sites.drop(columns=cols_to_drop)

## Create site AOIs

#### Add all custom site AOI geojsons here

In [5]:
custom_geojson_dict = {
    'PICS Libya-4': '/explore/nobackup/projects/CSDA_eval/sites/map_1x3_MODISgrids_Libya4.geojson'
}

In [6]:
# Create GeoDataFrame
sites_gdf = siteslib.create_sites_gdf_with_aois(
    sites, 
    default_size_km=BUF_KM,
    custom_geojson_dict=custom_geojson_dict, 
    site_column='Site Name',
    lat_column='Latitude',
    lon_column='Longitude'
)

# Calc area
sites_gdf['area_km2'] = siteslib.calculate_area_km2_per_site(sites_gdf)

print(f"\n{sites_gdf.shape[0]} sites are being used to search for imagery.")

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

✓ Loaded custom geometry for PICS Libya-4

115 sites are being used to search for imagery.


#### Remove misc fields

In [7]:
rm_cols_list = ['Site min sqkm', 'Max View Angle',
       'Min # of Acqs/Sensor', 'Max Cloud %', 'acquisition sqkm', 'Notes',
       'UL (Latitude, Longitude)', 'LL (Latitude, Longitude)',
       'UR (Latitude, Longitude)', 'LR (Latitude, Longitude)']

sites_gdf.drop(rm_cols_list, axis=1, inplace=True)

#### Save sites `GeoJSON` and copy to `GitHub` repo

In [8]:
csda_sites_fn = f'/explore/nobackup/projects/CSDA_eval/sites/csda_sites_aoi.geojson'
csda_sites_fn

'/explore/nobackup/projects/CSDA_eval/sites/csda_sites_aoi.geojson'

In [9]:
sites_gdf.to_file(csda_sites_fn)

In [10]:
!eval cp $csda_sites_fn /home/pmontesa/code/csda_summaries/sites/

#### Comfigure site AOIs for display

In [13]:
### Buffer MORE for sites for display
BUF_KM_ADD_FOR_DISPLAY = 25
BUF_KM_TOTAL_FOR_DISPLAY = BUF_KM_ADD_FOR_DISPLAY + BUF_KM

In [14]:
sites_gdf_buf = sites_gdf.to_crs(3857).buffer(BUF_KM_ADD_FOR_DISPLAY * 1000) # add arbitrary buffer to evaluation sites for map display
sites_gdf_buf_display = gpd.GeoDataFrame(sites_gdf.drop(columns=['geometry']), geometry=sites_gdf_buf, crs=sites_gdf_buf.crs).to_crs(4326)
sites_gdf_buf_display = sites_gdf_buf_display[~(sites_gdf_buf_display.geometry.is_empty | sites_gdf_buf_display.geometry.isna())]

### Check site AOIs (shown in `red`)

In [15]:
sites_gdf.explore(m=sites_gdf_buf_display.explore(color='black'), color='red')